# Groepsindeling 8×8 met vaste schaal ×64 / A32 — honderd epochs

Deze proef vergelijkt **8×8-groepen met de bestaande 1×32- en 1×64-groepen**, steeds met vaste ×64 voor de lagere precisies. De matrix van iedere MX-laag heeft outputkanalen op de rijen en afgevlakte `Cin × Kh × Kw`-posities op de kolommen. Een groep bevat acht rijen × acht kolommen: 64 gewichten met één E8M0-bronschaal. Beide matrixassen worden waar nodig met positieve nullen aangevuld.

E4M3 wordt uit de actuele effectieve FP32-gewichten G(W) bepaald. E2M3 behoudt de hoogste twee exponentbits en gebruikt **E8M0-broncode +6**. E2M1 verwijdert daarna twee mantissabits van E2M3 en behoudt dezelfde E2M3-schaal. Eén E4M3-bron wordt per laag en minibatch gedeeld door de drie modi. Activaties zijn **A32**; de updatemaster, gradients en Adam-momenten zijn **FP32**. Er zijn geen geleerde schaalparameters.

**De hoofdfiguren tonen epoch 100 en het gezamenlijke checkpoint gekozen door de hoogste E4M3-accuracy.** Alle drie scores komen steeds uit dezelfde epoch. De volledige accuracycurves vergelijken de drie groepsindelingen over epochs 1–100. Bij een onvoltooide run blijven alleen de gezamenlijke gemeten epochs zichtbaar; epoch 100 wordt niet als eindresultaat ingevuld.

**Nieuwe initialisatie:** deze honderd-epochtraining begint opnieuw met dezelfde seed en oorspronkelijke modelstaat. De geslaagde vijf-epochpilot is afzonderlijk bewaard in `pilot_analysis`; de pilotepochs worden niet aan deze run toegevoegd en de pilotcheckpoint wordt niet hervat.

De kostenanalyse toont apart het aantal schalen, de padding en de totale code- en schaalbytes voor 1×32, 1×64 en 8×8. Dit notebook leest uitsluitend opgeslagen JSON/CSV en rekent geometrische opslagkosten na. Het laadt geen dataset, model, checkpoint of optimizer en start geen training.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runkopie bevat hier het exacte resultaatpad.
SAVE_FIGURES = False  # True schrijft naar een nieuwe analysis_<tijdstip>-map.


In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

# Gebruik de helper uit déze run, ook als een eerdere notebook dezelfde modulenaam gebruikte.
helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
helper_name = '_mx_triple_group8x8_scale64_reference100_analysis_helper'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
if spec is None or spec.loader is None:
    raise FileNotFoundError(f'Analysehelper ontbreekt: {helper_path}')
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(Markdown('### Meldingen en ontbrekende gegevens'))
display(ns.diagnostics_table(run))


## Twee checkpointkeuzes: epoch 100 en het beste E4M3-checkpoint

**Epoch 100** is het vooraf gekozen vaste eindpunt. Het **beste checkpoint** is de eerste epoch met de hoogste E4M3-testaccuracy binnen de beschikbare geschiedenis. E2M3 en E2M1 worden op datzelfde checkpoint getoond; zij kiezen geen eigen beste epoch.

De tabel bewaart daarnaast de laatste beschikbare epoch. Bij een gedeeltelijke of niet bevestigde volledige run blijft de epoch-100-rij leeg en toont de linkerfiguur de laatste werkelijk gemeten epoch, duidelijk als voorlopig eindpunt. Het beste checkpoint wordt dan uitsluitend binnen die beschikbare epochs gekozen.


In [ ]:
display(ns.checkpoint_table(run).round({'Testaccuracy [%]': 6, 'Harde test-CE': 8}))
figures['checkpoint_accuracy'] = ns.plot_checkpoint_accuracy(run)
display(figures['checkpoint_accuracy'])
plt.close(figures['checkpoint_accuracy'])


## Vergelijking met 1×32 en 1×64: accuracy over de volledige training

De vastgelegde 1×32- en 1×64-referenties bevatten hun voltooide honderd-epochtrainingen. De figuren vergelijken de **curves over epochs 1–100**, **epoch 100** en het **gezamenlijke beste E4M3-checkpoint**. Referentiehashes, oorspronkelijke modelinitialisatie en trainingsinstellingen worden gecontroleerd voordat scores naast elkaar verschijnen.

Bij een gedeeltelijke 8×8-run stoppen alle curves bij de laatste beschikbare 8×8-epoch. Ook de beste-checkpointkeuze wordt dan voor iedere training beperkt tot diezelfde gezamenlijke epochrange. De epoch-100-vergelijking verschijnt uitsluitend als de nieuwe 8×8-run volledig en consistent is afgerond. De eerdere vijf-epochpilot vormt een apart resultaat en wordt niet samengevoegd met deze nieuwe training.

**1×64 tegenover 8×8 vergelijkt hetzelfde aantal gewichten per gedeelde schaal met een andere groepsvorm.** De vergelijking met 1×32 verandert zowel de groepsvorm als het aantal gewichten per groep. De ×64-schaalroute blijft in alle gevallen gelijk. Per training komen alle drie scores uit de epoch die door E4M3 gekozen is.


In [ ]:
references = ns.load_group_references(run)
display(Markdown(ns.comparison_markdown(run, references)))
display(ns.comparison_table(run, references).round(6))
figures['group_comparison_curves'] = ns.plot_scale_comparison_curves(run, references)
display(figures['group_comparison_curves'])
plt.close(figures['group_comparison_curves'])
figures['group_comparison_checkpoints'] = ns.plot_scale_comparison_checkpoints(run, references)
display(figures['group_comparison_checkpoints'])
plt.close(figures['group_comparison_checkpoints'])


## Kosten: aantal schalen, padding en totale opslag

Elke groep heeft één E8M0-schaal van acht bits. Zonder padding kost de schaal 0,25 bits per gewicht bij 1×32 en 0,125 bits bij zowel 1×64 als 8×8. Bij 1D-groepen wordt iedere outputrij tot een veelvoud van 32 of 64 aangevuld; bij 8×8 worden beide matrixassen tot veelvouden van acht aangevuld.

De berekening gebeurt per laag en per precisie. Voor E4M3 tonen we de afzonderlijke overhead van schaalbytes en paddingcodes bovenop de reële gewichtswaarden. **Voor deze zes laagvormen heeft 8×8 slechts 64 paddingposities**, tegenover 1.088 bij 1×32 en 8.768 bij 1×64. Dit kostenverschil is vooraf exact uit de vormen berekenbaar en zegt nog niets over de accuracy.


In [ ]:
display(Markdown(ns.cost_markdown(run)))
display(ns.cost_summary_table(run).round(6))
figures['group_size_costs'] = ns.plot_group_costs(run)
display(figures['group_size_costs'])
plt.close(figures['group_size_costs'])


### Kosten per laag en controle tegen de opgeslagen exports

De eerste tabel geeft de berekende E4M3-bronindeling per laag voor 1×32, 1×64 en 8×8. De lagere precisies worden met dezelfde groepsgeometrie afzonderlijk berekend. `group_size_costs_layers.csv` bevat alle lagen en alle drie de precisies.

De tweede tabel controleert de **8×8-indeling** tegen de opgeslagen exportdiagnostiek op het beste en laatste checkpoint. Deze controle bevestigt gerapporteerde aantallen; de codec- en exporttests van de runner controleren daarnaast de werkelijke codebytes, padding op beide matrixassen en gereconstrueerde gewichten. Het label 64 gewichten per groep alleen volstaat niet om 1×64 van 8×8 te onderscheiden.


In [ ]:
display(ns.cost_layer_table(run, weight_bits=8))
display(ns.export_layout_table(run))


## Kwantisatiefout op de twee geselecteerde checkpoints

De opgeslagen exportdiagnostiek vergelijkt de gecodeerde gewichten met de effectieve FP32-gewichten G(W), zonder een nieuwe evaluatie te starten. RMSE wordt gewogen over de reële gewichten van de zes MX-lagen; padding telt niet mee. Nieuwe nulwaarden zijn gewichten die vóór kwantisatie niet nul waren en daarna nul worden. Dit is gewichtsdiagnostiek, geen vervanging voor testaccuracy.


In [ ]:
display(ns.quantization_table(run))
figures['quantization_diagnostics'] = ns.plot_quantization(run)
display(figures['quantization_diagnostics'])
plt.close(figures['quantization_diagnostics'])


## Train- en testaccuracy per modus

De drie modi worden gezamenlijk vanaf dezelfde oorspronkelijke initialisatie getraind. Trainaccuracy wordt tijdens de updates verzameld. Testaccuracy wordt na iedere epoch gemeten met **8×8-groepen** en dezelfde ×64-truncatieroute, bevroren evaluatiestaat en de volledige testset. De drie modusgebonden BatchNorm-banken leren tijdens de training normaal mee.


In [ ]:
figures['accuracy_curves'] = ns.plot_learning(run)
display(figures['accuracy_curves'])
plt.close(figures['accuracy_curves'])


## Losses: trainingsdoelen onderscheiden van harde test-CE

De trainpass van E4M3 gebruikt harde CE van labels. Daarna leert E2M3 met soft-target CE van de **detached E4M3-softmax**; E2M1 leert met soft-target CE van de **detached E2M3-softmax**. Dit is recursieve distillatie met T=1. De drie gradients worden gesommeerd zonder deling door drie, waarna één Adam-update volgt; er is geen extra FP32-teacherforward.

De drie trainobjectieven staan in afzonderlijke panelen omdat hun targets verschillen. Soft CE is geen harde label-CE en wordt hier ook niet als KL-divergentie aangeduid. **Testloss is voor alle drie modi harde CE van dezelfde labels** en staat daarom samen in één vergelijkbaar paneel.


In [ ]:
if not run.metrics.empty:
    objective_columns = [key for key in ('epoch', 'phase', 'weight_format', 'objective_type', 'objective_loss') if key in run.metrics]
    display(run.metrics[objective_columns].round({'objective_loss': 8}))
figures['objective_losses'] = ns.plot_objectives(run)
display(figures['objective_losses'])
plt.close(figures['objective_losses'])


## Blijft het netwerk verschillende cijfers voorspellen?

De eerdere post-hocproeven lieten bij sommige routes sterk vertekende of constante voorspellingen zien. Daarom wordt hier per testepoch en per modus bijgehouden hoe de voorspellingen over de tien cijfers verdeeld zijn. Een constante voorspelling is direct zichtbaar als **één gebruikte klasse en 100% concentratie**. De tabel toont ook hoe vaak het dominante cijfer werkelijk in de testset voorkomt.

Dit zijn geaggregeerde tellingen; deze notebook doet geen nieuwe inferentie en leidt er geen per-beeldvergelijking uit af.


In [ ]:
display(ns.prediction_table(run, selected_only=True))
figures['prediction_diversity'] = ns.plot_prediction_diversity(run)
display(figures['prediction_diversity'])
plt.close(figures['prediction_diversity'])


In [ ]:
display(Markdown('## Gemeten trainingsduur en CUDA-geheugen\n\nNa een bevestigde volledige run wordt de **gemeten som van de honderd epochtijden** getoond, met train, test, controles en checkpointschrijven afzonderlijk. Bij een gedeeltelijke run blijft een eventueel opgeslagen tijdprojectie duidelijk een schatting voor een nieuwe volledige run; er wordt geen ontbrekende accuracy voorspeld.\n\nDe geheugenfiguur betreft het werkelijke CUDA-allocatorgebruik van deze FP32-emulator. De logische MX-code- en schaalpayload staat afzonderlijk in de kostenanalyse.\n'))
display(Markdown(ns.timing_markdown(run)))
figures['timing'] = ns.plot_timing(run)
display(figures['timing'])
plt.close(figures['timing'])


In [ ]:
display(Markdown('## GPU-geheugen\n\nDe opgeslagen pieken betreffen de PyTorch CUDA-allocator. Zij meten niet alle GPU-processen of compacte native MX-uitvoering. Ontbrekende waarden blijven ontbrekend.'))
figures['gpu_memory'] = ns.plot_memory(run)
display(figures['gpu_memory'])
plt.close(figures['gpu_memory'])


In [ ]:
display(Markdown('## Opgeslagen technische controles\n\nDeze tabellen rapporteren bestaande controles uit de run. Het notebook voert geen nieuwe gradient-, model- of checkpointcontrole uit.'))
display(ns.audit_table(run))
display(Markdown('### Gewichtsupdates\n\nDe mastergewichten zijn dezelfde gedeelde tensors. De herhaalde masterkolommen per modus mogen **niet bij elkaar worden opgeteld**; code- en schaalwijzigingen behoren wel bij het betreffende formaat.'))
display(ns.weight_update_table(run))
display(Markdown('### Eén opgeslagen E4M3-bron en twee afgeleide exports per geselecteerd checkpoint'))
display(ns.export_table(run))


## Interpretatie en grenzen

De gerichte wijziging is **8×8-groepering** tegenover de bestaande 1×64- en 1×32-referenties. De factor blijft ×64, mantissatruncatie behoudt de E2M3-schaal, en de overige trainingsinstellingen en initiële mastergewichten komen overeen. De identity-STE geldt voor de volledige codebewerking naar G(W), zonder extra factor 64 in de backward.

Een 8×8-groep deelt de schaal tussen verschillende outputfilters. Dat kan de quantisatiefout en accuracy veranderen; minder padding garandeert geen betere accuracy. Een lage accuracy blijft een onderzoeksuitkomst wanneer de technische controles slagen. Bekijk testaccuracy, harde test-CE en voorspellingstellingen samen.

De hardwaremotivatie is dat een vierkant blok dezelfde 64 gewichten blijft bevatten bij transpositie. De codeccontrole toetst die eigenschap en de bijbehorende schalen. Deze training emuleert de convoluties in FP32 en bewijst op zichzelf geen snelheids-, transfer- of energiewinst. Zie Cuyckens et al., *Efficient Precision-Scalable Hardware for Microscaling (MX) Processing in Robotics Learning*, §IV-A: https://arxiv.org/html/2505.22404v2.

De oorspronkelijke cuDNN-backend garandeert geen bitexacte CUDA-herhaalbaarheid. Deze proef heeft één seed; een klein accuracyverschil bewijst geen algemene superioriteit of optimale groepsindeling. Het beste E4M3-checkpoint wordt op de officiële testset geselecteerd en is geen onafhankelijke testschatting na selectie. Het vaste eindpunt epoch 100 blijft daarom afzonderlijk zichtbaar.

**Kosten horen bij de conclusie:** rapporteer schaalbytes, padding en totale code- en schaalpayload afzonderlijk. De drie exportgroottes zijn alternatieven; één E4M3-bronmodel wordt niet driemaal geteld. De oorspronkelijke compacte exports bewaren ook FP32-uitzonderingen en alle vijf BN-banken; de code- en schaaltabellen hier sluiten die auxiliaire staat uit. De recente inference-transferproef telde alleen ondersteunde BN-banken. Logische payload, checkpointbestandsgrootte en gemeten GPU-geheugen zijn verschillende grootheden.

De vijf-epochpilot en de nieuwe honderd-epochtraining blijven afzonderlijke runs. Dit notebook interpreteert uitsluitend de bestanden in `RUN_DIR`; onvoltooide of ontbrekende resultaten worden nooit als een voltooide honderd-epochtraining voorgesteld.


In [ ]:
if SAVE_FIGURES:
    analysis_dir = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', analysis_dir)
else:
    print('Alleen analyse in het notebook; geen figuren naar schijf geschreven.')
print('Ingelezen resultaatmap:', run.path)
display(pd.DataFrame({'Ingelezen bestand': [str(path.relative_to(run.path)) if path.is_relative_to(run.path) else str(path) for path in run.input_files]}))
